In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from Helpers import Helpers
import re

spark = SparkSession.builder.appName("Lab2").master("local[*]").getOrCreate()
sc = spark.sparkContext

marsrutas = "marsrutas"
svoris = "svoris"
svorio_grupe = "svorio grupe"
geografine_zona = "geografine zona"
sustojimo_data = "sustojimo data"
siuntu_skaicius = "siuntu skaicius"
sustojimo_savaites_diena = "sustojimo savaites diena"
sustojimo_klientu_skaicius = "Sustojimo klientu skaicius"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/24 19:07:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Užduotis 1.

In [5]:
lines = sc.textFile("./data/duom_full.txt")
entries = lines.flatMap(lambda line: list(Helpers.extract_entries(line, [svoris, svorio_grupe])))

group_weight_pairs = entries.map(lambda entry: (entry[svorio_grupe], entry[svoris]))

grouped = group_weight_pairs.groupByKey()

def calculate_stats(weights):
    weights_list = list(weights)
    return (
        min(weights_list),
        max(weights_list),
        sum(weights_list) / len(weights_list)
    )

result = grouped.mapValues(calculate_stats)

formatted_results = result.map(lambda x: {
    "weight_group": x[0],
    "min_weight": x[1][0],
    "max_weight": x[1][1],
    "avg_weight": x[1][2]
})

results = formatted_results.collect()

# Convert to DataFrame for visualization
result_df = spark.createDataFrame(formatted_results)
result_df.show()

+------------------+----------+----------+------------+
|        avg_weight|max_weight|min_weight|weight_group|
+------------------+----------+----------+------------+
| 5.964402623894628|      50.0|       0.0|         <50|
| 759.2116686954745|   6896.65|    300.05|        >300|
|110.03327885846451|     300.0|     50.05|        <300|
+------------------+----------+----------+------------+



## Užduotis 2.

In [ ]:
lines = sc.textFile("./data/duom_full.txt")
entries = lines.flatMap(lambda line: list(Helpers.extract_entries(line, [marsrutas, geografine_zona, sustojimo_data])))

# Rasti unikalius maršrutus
all_routes = entries.map(lambda entry: entry[marsrutas]).distinct()
total_routes = all_routes.count()
if total_routes == 0:
    assert("Rasta 0 maršrutų.")

route_zone_date_pairs = entries.map(lambda entry: (
    entry[marsrutas], 
    (entry[geografine_zona], entry[sustojimo_data])
))

# Grupuojame pagal maršrutą ir randame unikalias zonas
def analyze_zones_and_dates(zone_date_pairs):
    unique_zones = set(zone for zone, _ in zone_date_pairs)
    
    zones_by_date = {}
    for zone, date in zone_date_pairs:
        if date not in zones_by_date:
            zones_by_date[date] = set()
        zones_by_date[date].add(zone)
    
    # Rasti dienas, kuriomis aplankyta daugiau nei viena zona
    multi_zone_dates = [date for date, zones in zones_by_date.items() if len(zones) > 1]
    
    return {
        "unique_zones": unique_zones,
        "zones_count": len(unique_zones),
        "has_multi_zones": len(unique_zones) > 1,
        "zones_by_date": zones_by_date,
        "multi_zone_dates": multi_zone_dates,
        "has_multi_zones_same_day": len(multi_zone_dates) > 0
    }

route_analysis = route_zone_date_pairs.groupByKey().mapValues(analyze_zones_and_dates)

# Rasti maršrutus, kurie aplanko daugiau nei 1 zoną
multi_zone_routes = route_analysis.filter(lambda x: x[1]["has_multi_zones"])
multi_zone_count = multi_zone_routes.count()

# Rasti maršrutus, kurie aplanko daugiau nei 1 zoną tą pačią dieną
multi_zone_same_day_routes = route_analysis.filter(lambda x: x[1]["has_multi_zones_same_day"])
multi_zone_same_day_count = multi_zone_same_day_routes.count()

# Rasti procentus
multi_zone_percent = (multi_zone_count / total_routes) * 100
multi_zone_same_day_percent = (multi_zone_same_day_count / total_routes) * 100

print(f"Iš viso unikalių maršrutų: {total_routes}")
print(f"Maršrutai, aplankantys daugiau nei vieną geografinę zoną: Skaičius: {multi_zone_count} Procentai: ({multi_zone_percent:.2f}%)")
print(f"Maršrutai, aplankantys daugiau nei vieną geografinę zoną tą pačią dieną: Skaičius: {multi_zone_same_day_count} Procentai: ({multi_zone_same_day_percent:.2f}%)")

same_day_results = multi_zone_same_day_routes.sortByKey().take(5)

for route, analysis in same_day_results:
    print(f"\nMaršrutas {route}:")
    for date, zones in analysis["zones_by_date"].items():
        if len(zones) > 1:
            print(f"  Data: {date}, Aplankytos zonos: {', '.join(zones)}")

Iš viso unikalių maršrutų: 424
Maršrutai, aplankantys daugiau nei vieną geografinę zoną: Skaičius: 395 Procentai: (93.16%)
Maršrutai, aplankantys daugiau nei vieną geografinę zoną tą pačią dieną: Skaičius: 394 Procentai: (92.92%)



Maršrutas 32.0:
  Data: 2018-01-02, Aplankytos zonos: Z3, Z1
  Data: 2018-01-29, Aplankytos zonos: Z3, Z1

Maršrutas 33.0:
  Data: 2018-01-16, Aplankytos zonos: Z3, Z1
  Data: 2018-01-31, Aplankytos zonos: Z3, Z1

Maršrutas 34.0:
  Data: 2018-01-05, Aplankytos zonos: Z3, Z1
  Data: 2018-01-09, Aplankytos zonos: Z3, Z1

Maršrutas 35.0:
  Data: 2018-01-11, Aplankytos zonos: Z3, Z1
  Data: 2018-01-22, Aplankytos zonos: Z3, Z1
  Data: 2018-01-23, Aplankytos zonos: Z3, Z1

Maršrutas 36.0:
  Data: 2018-01-08, Aplankytos zonos: Z3, Z1
  Data: 2018-01-31, Aplankytos zonos: Z3, Z1


## Užduotis 3.

In [4]:
from collections import defaultdict
lines = sc.textFile("./data/duom_full.txt")

entries = lines.flatMap(lambda line: list(Helpers.extract_entries(line, [
    geografine_zona,
    sustojimo_savaites_diena,
    siuntu_skaicius,
    sustojimo_klientu_skaicius
])))

# Sukuriame rakto-reikšmės poras: (zona, diena) -> (siuntų skaičius, klientų skaičius)
zone_day_pairs = entries.map(lambda entry: (
    (entry[geografine_zona], entry[sustojimo_savaites_diena]),
    (entry[siuntu_skaicius], entry[sustojimo_klientu_skaicius])  
))


zone_day_sums = zone_day_pairs.reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))

results = zone_day_sums.map(lambda x: {
    "geografine_zona": x[0][0],
    "savaites_diena": int(x[0][1]),
    "pristatyta_siuntiniu": int(x[1][0]),
    "aptarnauta_klientu": int(x[1][1])
})

result_df = spark.createDataFrame(results)
result_df = result_df.orderBy("geografine_zona", "savaites_diena")

result_df.show()

+------------------+---------------+--------------------+--------------+
|aptarnauta_klientu|geografine_zona|pristatyta_siuntiniu|savaites_diena|
+------------------+---------------+--------------------+--------------+
|             41580|             Z1|               86464|             1|
|             60167|             Z1|              123938|             2|
|             62241|             Z1|              126872|             3|
|             48430|             Z1|               98156|             4|
|             45999|             Z1|               88663|             5|
|               797|             Z1|                1754|             6|
|             11751|             Z2|               19525|             1|
|             18499|             Z2|               31377|             2|
|             18545|             Z2|               30715|             3|
|             14190|             Z2|               23727|             4|
|             14027|             Z2|               